# House Price Data — Complete Preprocessing Pipeline

---

**Project:** ML with Scikit-Learn  
**Notebook:** 01 — Data Cleaning and Preprocessing  
**Dataset:** housing_sample.csv.txt  
**Author:** Ather-ops  

---

## Objective

This notebook focuses exclusively on **data cleaning and preprocessing** — transforming raw, messy housing data into a clean, structured, ML-ready dataset. No model training happens here. The output of this notebook is a fully prepared feature matrix that can be passed directly into any Scikit-Learn model.

---

## What This Notebook Covers

| Step | Task | Purpose |
|------|------|---------|
| 1 | Import libraries | Load all required tools |
| 2 | Load dataset | Read CSV, inspect raw structure |
| 3 | Exploratory visualization | Understand feature-target relationships visually |
| 4 | Descriptive statistics | Summarize distributions and spot anomalies |
| 5 | Missing value check | Detect NaN values that break ML models |
| 6 | Duplicate detection | Identify and remove repeated rows |
| 7 | Outlier detection (IQR) | Find and clip extreme price values |
| 8 | Outlier treatment | Apply clipping to clean the Price column |
| 9 | Feature binning by Area | Group continuous Area into size categories |
| 10 | Mean price analysis per bin | Understand average price per size group |
| 11 | Feature binning by Age | Group continuous Age into lifecycle categories |
| 12 | One-Hot Encoding | Convert categorical bins into numeric feature vectors |
| 13 | Feature scaling | Normalize features to equal scale |
| 14 | Final cleaned dataset | Inspect and confirm the ML-ready output |

---

## Why Preprocessing Matters

Raw data is almost never fit for a machine learning model. Common problems include:

- Missing values that cause runtime errors during training
- Duplicate rows that bias the model toward overrepresented patterns
- Outliers that distort learned weights and inflate error metrics
- Features on different scales that cause gradient descent to converge poorly
- Categorical text labels that numeric models cannot process

Every step in this notebook addresses one of these problems. A model trained on clean data consistently outperforms one trained on raw data — preprocessing is where most of the real impact happens.

---
## Step 1 — Import Libraries

We import only what is needed for preprocessing. Model-related imports such as LinearRegression and train_test_split are intentionally excluded — this notebook ends before model training begins.

| Library | Role in this notebook |
|---------|----------------------|
| `pandas` | Load CSV, inspect data, binning, encoding, groupby operations |
| `numpy` | Numerical operations, array handling |
| `matplotlib.pyplot` | Visualize raw distributions and feature-target relationships |
| `StandardScaler` | Normalize numerical features to mean=0, std=1 |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler

---
## Step 2 — Load Dataset

We read the housing CSV into a Pandas DataFrame and inspect the first 10 rows. This is always the first action with any new dataset — it reveals the structure, column names, data types, and any immediately visible issues.

**Expected columns:**

| Column | Data Type | Description |
|--------|-----------|-------------|
| `Area` | int / float | Size of the house in square feet |
| `Bedrooms` | int | Number of bedrooms |
| `Age` | int | Age of the house in years |
| `Price` | int / float | Sale price of the house — this is the target variable |

After printing the head, check:
- Are column names as expected?
- Do the values look reasonable?
- Are any columns obviously wrong (e.g. text where numbers should be)?

In [ ]:
df = pd.read_csv("housing_sample.csv.txt")
print("="*50)
print("original data:\n", df.head(10))

---
## Step 3 — Dataset Shape and Data Types

Before doing anything else, we check the full shape of the dataset and confirm each column has the correct data type.

**Why this matters:**
- `df.shape` tells us how many rows and columns we are working with
- `df.dtypes` confirms whether columns are numeric (`int64`, `float64`) or text (`object`)
- If `Area` or `Price` shows as `object`, it means some values contain non-numeric characters and must be cleaned before any numeric operations can run
- `df.info()` gives a combined view of shape, dtypes, and non-null counts in one call

In [ ]:
# Dataset shape and column info
print("Dataset shape:", df.shape)
print("="*50)
print("Column data types:")
print(df.dtypes)
print("="*50)
print("Full info:")
df.info()

---
## Step 4 — Exploratory Visualization

Before any cleaning or transformation, we visualize the raw relationships between each input feature and the target variable `Price`. This is called **Exploratory Data Analysis (EDA)**.

**Why visualize raw data first?**

- Confirms whether a linear relationship exists between features and target
- Reveals outliers as isolated dots far from the main cluster
- Shows the spread and density of each feature
- Helps decide which features are worth keeping and which may need transformation

**What to look for in each plot:**

| Plot | Expected Pattern | What deviation means |
|------|-----------------|---------------------|
| Area vs Price | Upward trend — bigger house costs more | If flat, Area may not be a useful feature |
| Bedrooms vs Price | Mild upward trend | Large scatter is normal — bedrooms alone don't determine price |
| Age vs Price | Downward trend — older house costs less | If flat, Age adds little predictive value |

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.scatter(df["Area"], df["Price"], color="red")
plt.xlabel("Area")
plt.ylabel("Price")
plt.title("AREA VS PRICE(Before Training)")

plt.subplot(1,3,2)
plt.scatter(df["Bedrooms"], df["Price"], color="yellow")
plt.xlabel("No of bedrooms")
plt.ylabel("Price")
plt.title("BEDROOMS VS PRICE(Before Training)")

plt.subplot(1,3,3)
plt.scatter(df["Age"], df["Price"], color="green")
plt.xlabel("Age")
plt.ylabel("Price")
plt.title("AGE VS PRICE (Before Training)")

plt.tight_layout()
plt.show()

---
## Step 5 — Descriptive Statistics

`df.describe()` produces a statistical summary of every numeric column. Reading this table carefully reveals problems that are invisible in the raw data.

**What each row means:**

| Statistic | Meaning | What to check |
|-----------|---------|---------------|
| `count` | Non-null row count | If count differs across columns, some have missing values |
| `mean` | Average value | Should be plausible for the domain |
| `std` | Standard deviation — spread of values | Very high std relative to mean suggests outliers |
| `min` | Smallest value | Negative prices or zero area would be data errors |
| `25%` | First quartile Q1 | Bottom 25% of values |
| `50%` | Median | Middle value — resistant to outliers unlike mean |
| `75%` | Third quartile Q3 | Top 25% begins here |
| `max` | Largest value | An unusually large max compared to the 75th percentile indicates an outlier |

**Red flags to look for:**
- `min` is negative for a column that should never be negative (e.g. Price, Area)
- `max` is many times larger than `75%` — strong sign of an outlier
- `mean` is much larger than `median` (`50%`) — distribution is right-skewed

In [ ]:
print("Basic statistic:\n", df.describe())
print("missing values:\n", df.isnull().sum())
print("="*50)

---
## Step 6 — Missing Value Analysis

Missing values (`NaN`) cause silent failures or runtime errors in Scikit-Learn. We inspect them in detail and apply a treatment strategy.

**Detection:**
- `df.isnull().sum()` — count of missing values per column (already printed above)
- `df.isnull().sum() / len(df) * 100` — percentage of missing values per column

**Treatment strategies:**

| Strategy | When to use |
|----------|-------------|
| Drop the row | Less than 5% of rows are affected and the dataset is large |
| Fill with mean | Column is numeric and roughly normally distributed |
| Fill with median | Column is numeric but skewed or has outliers |
| Fill with mode | Column is categorical |
| Forward fill | Time series data where the previous value is a logical substitute |

For this housing dataset we apply **median imputation** for numeric columns — median is more robust than mean when outliers are present.

In [ ]:
# Detailed missing value report
missing_count = df.isnull().sum()
missing_pct   = df.isnull().sum() / len(df) * 100
missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_pct.round(2)
})
print("Missing Value Report:")
print("="*50)
print(missing_report)
print("="*50)

# Fill numeric columns with median
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled '{col}' missing values with median: {median_val}")

print("\nMissing values after treatment:", df.isnull().sum().sum())
print("="*50)

---
## Step 7 — Duplicate Row Detection and Removal

Duplicate rows occur when the same house entry is recorded more than once — this can happen from data entry errors, system imports, or merging multiple data sources.

**Why duplicates are harmful:**
- The model sees the same data point multiple times and treats it as stronger evidence than it actually is
- This biases learned weights toward overrepresented patterns
- Evaluation metrics appear better than they really are

**How to handle them:**
- `df.duplicated().sum()` — count total duplicate rows
- `df.drop_duplicates()` — keep the first occurrence, remove all subsequent ones
- Always print the shape before and after to confirm how many rows were removed

In [ ]:
# Check for duplicate rows
print("Shape before duplicate removal:", df.shape)
print("Duplicate rows found:", df.duplicated().sum())
print("="*50)

# Remove duplicates
df = df.drop_duplicates()
df = df.reset_index(drop=True)

print("Shape after duplicate removal:", df.shape)
print("Dataset is clean of duplicates.")
print("="*50)

---
## Step 8 — Outlier Detection using the IQR Method

Outliers are extreme values that lie far outside the normal range of the data. A single outlier — for example a house priced at 10x the market average — can distort the entire regression line and inflate error metrics dramatically.

**The IQR Method:**

The Interquartile Range is the spread of the middle 50% of the data:

$$IQR = Q3 - Q1$$

Safe boundaries (fences) are calculated as:

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Any value outside these fences is an outlier.

| Variable | Meaning |
|----------|---------|
| Q1 | 25th percentile — 25% of prices are below this |
| Q2 (Q3) | 75th percentile — 75% of prices are below this |
| IQR | Width of the middle 50% of the distribution |
| Lower Bound | Minimum acceptable price |
| Upper Bound | Maximum acceptable price |

In [ ]:
# step 4: Finding outlier
Q1=df["Price"].quantile(0.25)
Q2=df["Price"].quantile(0.75)
IQR=Q2-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q2+1.5*IQR
print("lower_bound:",lower_bound)
print("upper_bound:",upper_bound)
print("="*50)

---
## Step 9 — Outlier Treatment: Clipping

Now that we have the IQR fences, we apply the treatment. There are two common approaches:

| Approach | Method | When to use |
|----------|--------|-------------|
| Deletion | Remove rows where Price is outside fences | Only if outliers are genuine data errors and dataset is large |
| Clipping | Cap values at the fence boundaries | Preferred — preserves sample size, removes distortion |

We use **clipping** because:
- It keeps every row in the dataset
- It replaces extreme values with the nearest valid boundary rather than deleting them
- It is the industry-standard approach for tabular regression problems

`numpy.clip(value, lower, upper)` replaces any value below `lower` with `lower` and any value above `upper` with `upper`.

After clipping, we visualize the Price distribution to confirm the outliers have been removed.

In [ ]:
# Count outliers before clipping
outliers_before = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)].shape[0]
print(f"Outliers detected in Price column: {outliers_before}")
print("="*50)

# Apply clipping
df["Price"] = np.clip(df["Price"], lower_bound, upper_bound)

# Confirm treatment
outliers_after = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)].shape[0]
print(f"Outliers remaining after clipping: {outliers_after}")
print("="*50)

# Visualize Price distribution before and after — using the already-clipped data
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(df["Price"], bins=15, color="steelblue", edgecolor="white")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.title("Price Distribution After Clipping")
plt.axvline(lower_bound, color="red",  linestyle="--", label="Lower Fence")
plt.axvline(upper_bound, color="green", linestyle="--", label="Upper Fence")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.boxplot(df["Price"], vert=True, patch_artist=True,
            boxprops=dict(facecolor="steelblue", color="navy"),
            medianprops=dict(color="yellow", linewidth=2))
plt.ylabel("Price")
plt.title("Price Boxplot After Outlier Clipping")
plt.grid(True)

plt.tight_layout()
plt.show()

---
## Step 10 — Feature Binning by Area

**Binning** (also called bucketing) divides a continuous numerical column into discrete, labeled categories. This technique adds structured human-interpretable groupings to the dataset.

We divide `Area` into three size categories:

| Bin Range (sq ft) | Label | Property type |
|-------------------|-------|---------------|
| 0 to 1000 | `small_house` | Compact flat or studio |
| 1000 to 3000 | `medium_house` | Standard family home |
| 3000 to 5000 | `large_house` | Spacious or premium property |

**How `pd.cut()` works:**
- The `bins` list defines the boundary points
- The `labels` list assigns a category name to each interval
- Each row gets the label corresponding to the bin its Area falls into

**Important:** The resulting `Area_bins` column is **categorical**, not numerical. It cannot be used directly in a regression model — it will be converted to numeric via One-Hot Encoding in Step 12.

In [ ]:
# step 5: BINNING /BUCTING by AREA
bins=[0,1000,3000,5000]
labels=["small_house", "medium_house", "large_house"]
df["Area_bins"] = pd.cut(df["Area"], bins=bins, labels=labels)
print(df[["Area", "Area_bins"]])
print("="*50)

---
## Step 11 — Mean Price per Area Bin

We calculate the average price within each size category using `groupby().transform('mean')`. This serves two purposes:

1. **Validation** — confirms that our binning makes intuitive sense (larger bins should have higher average prices)
2. **Feature engineering** — `mean_price_by_bin` is a form of **target encoding** that can be used as an additional feature in the model

**Difference between `transform()` and `mean()`:**

| Method | Output | Shape |
|--------|--------|-------|
| `groupby().mean()` | Aggregated table — one row per group | Smaller than original |
| `groupby().transform('mean')` | Group mean assigned to every original row | Same shape as original |

We use `transform()` because we want the group average aligned row-by-row with the rest of the DataFrame — not a collapsed summary table.

In [ ]:
# step 6: Mean of house bins
df["mean_price_by_bin"]=df.groupby("Area_bins")["Price"].transform("mean")
print(df[["Area_bins","Price","mean_price_by_bin"]])
print("="*50)

---
## Step 11b — Average Price per Bin — Visualization

We plot the mean price per Area bin as a bar chart to visually confirm the expected price gradient across size categories. This is a quick sanity check — if `small_house` showed a higher mean price than `large_house`, there would be a data problem worth investigating.

In [ ]:
# Bar chart of mean price per area bin
bin_means = df.groupby("Area_bins")["Price"].mean()

plt.figure(figsize=(7, 4))
plt.bar(bin_means.index.astype(str), bin_means.values,
        color=["steelblue", "darkorange", "green"], edgecolor="white", width=0.5)
plt.xlabel("Area Bin")
plt.ylabel("Mean Price")
plt.title("Mean Price by Area Bin")
plt.grid(axis="y", linestyle="--", alpha=0.6)
for i, v in enumerate(bin_means.values):
    plt.text(i, v + 500, f"{v:,.0f}", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

---
## Step 12 — Feature Binning by Age

We apply the same binning approach to the `Age` column, creating three lifecycle categories.

| Bin Range (years) | Label | Interpretation |
|-------------------|-------|----------------|
| 0 to 3 | `New_house` | Recently built — typically commands premium pricing |
| 3 to 6 | `Moderate` | Mid-age property — standard depreciation applied |
| 6 to 10 | `old_house` | Older property — more depreciated, lower expected price |

**Why bin Age rather than use it as a raw number?**

The price decrease from year 1 to year 3 is not necessarily the same as from year 7 to year 10. Binning captures non-linear depreciation patterns in a way that a linear coefficient on raw `Age` cannot. The model treats `New_house`, `Moderate`, and `old_house` as distinct groups with their own price levels rather than assuming a fixed price drop per year.

In [ ]:
# step 7: BINNING /BUCKTING BY AGE
bins=[0,3,6,10]
labels=["New_house","Moderate","old_house"]
df["Age_bins"]=pd.cut(df["Age"],bins=bins,labels=labels)
print(df[["Area_bins","Age","Age_bins"]])
print("="*50)

---
## Step 13 — One-Hot Encoding

Machine learning models are mathematical — they cannot process text labels like `small_house` or `New_house`. **One-Hot Encoding** converts each category into a separate binary (0 or 1) column.

**How it works:**

For `Area_bins` with 3 categories, encoding creates 3 new columns:

| Original Area_bins | small_house | medium_house | large_house |
|--------------------|-------------|--------------|-------------|
| small_house | 1 | 0 | 0 |
| medium_house | 0 | 1 | 0 |
| large_house | 0 | 0 | 1 |

For `Age_bins` with 3 categories, encoding creates 3 more new columns:

| Original Age_bins | New_house | Moderate | old_house |
|-------------------|-----------|----------|-----------|
| New_house | 1 | 0 | 0 |
| Moderate | 0 | 1 | 0 |
| old_house | 0 | 0 | 1 |

`pd.get_dummies()` performs this conversion automatically.  
`pd.concat(..., axis=1)` attaches the new columns horizontally to the existing DataFrame.

**Note on the Dummy Variable Trap:**

With 3 categories, having all 3 binary columns creates perfect multicollinearity — any one column can be perfectly predicted from the other two. In a strict Linear Regression setting you would drop one column per group (`drop_first=True`). For exploratory preprocessing we keep all columns to retain full transparency.

In [ ]:
# step 8: One-Hot Encoder
# converting area and age bins into feature vector
one_hot_area_bins=pd.get_dummies(df["Area_bins"])
one_hot_age_bins=pd.get_dummies(df["Age_bins"])
# combine all into one hot column
df=pd.concat([df,one_hot_area_bins,one_hot_age_bins],axis=1)
print("\nDataFrame after all one-hot encoding:")
print("="*150)
print(df.head(10))

---
## Step 15 — Feature and Target Selection

Before scaling we need to define the feature matrix `X` and the target vector `Y`, then split into train and test sets.

We use the three core numerical features for `X`:

| Variable | Columns | Role |
|----------|---------|------|
| `X` | Area, Bedrooms, Age | Input to the model — what it learns from |
| `Y` | Price | What the model predicts |

**Train-Test Split:**

| Parameter | Value | Meaning |
|-----------|-------|--------|
| `test_size=0.2` | 20% | Held-out set for final evaluation |
| `random_state=42` | 42 | Fixed seed — ensures same split on every run |

The split is performed here as part of preprocessing so that StandardScaler can be applied correctly (fit only on `X_train`, transform both). The model training itself happens in a separate notebook.

In [ ]:
from sklearn.model_selection import train_test_split

# Feature matrix and target vector
X = df[["Area", "Bedrooms", "Age"]]
Y = df["Price"]

# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("X shape      :", X.shape)
print("Y shape      :", Y.shape)
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape :", Y_test.shape)
print("="*50)

---
## Step 16 — Apply StandardScaler Correctly

Now we apply the scaler using the proper fit-on-train-only rule. After scaling, we verify that the training features have been normalized correctly by printing mean and standard deviation.

In [ ]:
# Fit scaler on training data only — then apply to both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Verify scaling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=["Area", "Bedrooms", "Age"])

print("Scaled X_train — Mean per feature (should be close to 0):")
print(X_train_scaled_df.mean().round(6))
print("="*50)
print("Scaled X_train — Std per feature (should be close to 1):")
print(X_train_scaled_df.std().round(6))
print("="*50)
print("Scaler learned means :", scaler.mean_.round(4))
print("Scaler learned scales:", scaler.scale_.round(4))
print("="*50)

---
## Step 17 — Final Cleaned Dataset Inspection

We do a final inspection of the processed DataFrame to confirm that all cleaning steps have been applied correctly before this data is passed to a model.

**Checklist before passing to a model:**

| Check | Expected result |
|-------|-----------------|
| Missing values | 0 across all columns |
| Duplicate rows | 0 |
| Price outliers | All values within IQR fences |
| Categorical columns | Converted to numeric via one-hot encoding |
| Feature scale | Verified mean near 0, std near 1 after scaling |

In [ ]:
print("FINAL DATASET INSPECTION")
print("="*50)
print("Shape                 :", df.shape)
print("Missing values total  :", df.isnull().sum().sum())
print("Duplicate rows        :", df.duplicated().sum())
print("Price min             :", df["Price"].min())
print("Price max             :", df["Price"].max())
print("Price lower bound     :", round(lower_bound, 2))
print("Price upper bound     :", round(upper_bound, 2))
print("="*50)
print("Columns in final DataFrame:")
print(df.columns.tolist())
print("="*50)
print("Sample of final cleaned data:")
print(df.head(5))
print("="*50)
print("Preprocessing complete. X_train_scaled and X_test_scaled are ML-ready.")

---
## Preprocessing Pipeline Summary

This notebook has completed the full data cleaning and preprocessing pipeline for the housing dataset. Below is a summary of every transformation applied and its purpose.

| Step | Transformation | Technique | Output |
|------|---------------|-----------|--------|
| 2 | Load raw CSV | `pd.read_csv()` | Raw DataFrame |
| 3 | Inspect shape and types | `df.shape`, `df.dtypes`, `df.info()` | Structural overview |
| 4 | Visualize feature-target relationships | `plt.scatter()` | 3 scatter plots |
| 5 | Descriptive statistics | `df.describe()` | Statistical summary |
| 6 | Missing value detection and treatment | `df.isnull()`, median imputation | Zero NaN columns |
| 7 | Duplicate detection and removal | `df.drop_duplicates()` | Clean row count |
| 8 | Outlier detection | IQR method — Q1, Q3, fences | Bound values printed |
| 9 | Outlier treatment | `np.clip()` to fence boundaries | Price column clipped |
| 10 | Area binning | `pd.cut()` — 3 size categories | `Area_bins` column |
| 11 | Mean price per bin | `groupby().transform('mean')` | `mean_price_by_bin` column |
| 11b | Bin price visualization | `plt.bar()` | Bar chart |
| 12 | Age binning | `pd.cut()` — 3 lifecycle categories | `Age_bins` column |
| 13 | One-Hot Encoding | `pd.get_dummies()` + `pd.concat()` | 6 new binary columns |
| 14-16 | Feature selection, split, scaling | `train_test_split`, `StandardScaler` | `X_train_scaled`, `X_test_scaled` |

---

**Output passed to next notebook (02_linear_regression.ipynb):**

| Object | Contents |
|--------|----------|
| `X_train_scaled` | Scaled training features — ready for model.fit() in next notebook |
| `X_test_scaled` | Scaled test features — ready for model.predict() |
| `Y_train` | Training target prices |
| `Y_test` | Test target prices for evaluation |
| `scaler` | Fitted StandardScaler — needed to scale new input at inference time |